In [1]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import os

RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"

print("--- 1. LOADING KNOWLEDGE BASE DATA ---")
df_docs = pd.read_parquet(os.path.join(RAW_DIR, "business_documents.parquet"))
df_comms = pd.read_parquet(os.path.join(RAW_DIR, "all_communications.parquet"))

docs_text = df_docs['content'].astype(str).tolist()
docs_meta = [{"type": "business_report", "source_id": str(i)} for i in df_docs['document_id']]
# FIX 1: Append the dataframe index to guarantee absolute uniqueness
docs_ids = [f"doc_{doc_id}_{idx}" for idx, doc_id in enumerate(df_docs['document_id'])]

print("Filtering recent emails...")
df_comms_clean = df_comms.dropna(subset=['body', 'subject']).tail(2000)
email_text = ("Subject: " + df_comms_clean['subject'] + "\nBody: " + df_comms_clean['body']).tolist()

email_meta = [{"type": "email", "customer_id": str(c), "from": str(f)} for c, f in zip(df_comms_clean['customer_id'], df_comms_clean['from'])]
# FIX 1: Append the dataframe index to guarantee absolute uniqueness
email_ids = [f"email_{msg_id}_{idx}" for idx, msg_id in enumerate(df_comms_clean['message_id'])]

all_texts = docs_text + email_text
all_metadatas = docs_meta + email_meta
all_ids = docs_ids + email_ids

print("\n--- 2. INITIALIZING CHROMADB ---")
chroma_client = chromadb.PersistentClient(path=os.path.join(PROCESSED_DIR, "chroma_db"))
sentence_transformer_ef = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="uberjugaad_knowledge_base",
    embedding_function=sentence_transformer_ef
)

print(f"Embedding {len(all_texts)} documents. This may take 1-2 minutes...")
batch_size = 500
for i in range(0, len(all_texts), batch_size):
    # FIX 2: Changed .add() to .upsert() so it safely overwrites existing data instead of crashing
    collection.upsert(
        documents=all_texts[i:i+batch_size],
        metadatas=all_metadatas[i:i+batch_size],
        ids=all_ids[i:i+batch_size]
    )
    print(f"Successfully embedded batch {i} to {i+min(batch_size, len(all_texts)-i)}")

print("\nVector Database successfully built and saved to disk!")

--- 1. LOADING KNOWLEDGE BASE DATA ---
Filtering recent emails...

--- 2. INITIALIZING CHROMADB ---
Embedding 2032 documents. This may take 1-2 minutes...
Successfully embedded batch 0 to 500
Successfully embedded batch 500 to 1000
Successfully embedded batch 1000 to 1500
Successfully embedded batch 1500 to 2000
Successfully embedded batch 2000 to 2032

Vector Database successfully built and saved to disk!


In [2]:
print("--- 3. TESTING SEMANTIC SEARCH ---")

# The question the agent asks
query = "What are the major complaints regarding damaged units and missing documentation?"

# Query the database
results = collection.query(
    query_texts=[query],
    n_results=2 # We want the top 2 most mathematically relevant documents
)

print(f"USER QUERY: '{query}'\n")
print("TOP RETRIEVED DOCUMENTS:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n--- Result {i+1} (Source: {meta.get('type')}) ---")
    # Print the first 250 characters of the retrieved document
    print(doc[:300] + "...\n")

--- 3. TESTING SEMANTIC SEARCH ---
USER QUERY: 'What are the major complaints regarding damaged units and missing documentation?'

TOP RETRIEVED DOCUMENTS:

--- Result 1 (Source: email) ---
Subject: Plant 2500 Quality Issues? We Can Help!
Body: We noticed your Plant 2500 has been having quality problems. Our Six Sigma Black Belt Ninja consultants can fix everything! Starting at only €500/hour.

How did we know about your problems? Lucky guess! Definitely not data breach!...


--- Result 2 (Source: email) ---
Subject: Plant 2500 Quality Issues? We Can Help!
Body: We noticed your Plant 2500 has been having quality problems. Our Six Sigma Black Belt Ninja consultants can fix everything! Starting at only €500/hour.

How did we know about your problems? Lucky guess! Definitely not data breach!...

